In [0]:
# #CREATE weather_documents table:

# from lakebase import get_connection

# with open("sql/01_setup_weather.sql", "r") as f:
#     sql = f.read()

# with get_connection() as conn:
#     with conn.cursor() as cur:
#         cur.execute(sql)
#     conn.commit()

# print("weather_documents table created")

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
import importlib.util
# Delete the old module from sys.modules
sys.modules.pop("weather_client", None)
sys.modules.pop("lakebase_client", None)

In [0]:
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = Path(
    "/Workspace/Users/tuvu.uwyo@gmail.com/weather_intelligence_databricks"
)

def load_module(module_name):
    path = REPO_ROOT / f"{module_name}.py"

    spec = importlib.util.spec_from_file_location(
        module_name,
        path
    )
    module = importlib.util.module_from_spec(spec)

    # Make imports inside the module work
    sys.modules[module_name] = module
    spec.loader.exec_module(module)

    return module

lakebase_client = load_module("lakebase")
weather_module = load_module("weather")
weather_client = weather_module.WeatherClient()

print("weather_client loaded:", weather_module.__file__)
print("Has upsert_documents:", hasattr(weather_module, "upsert_documents"))

In [0]:
# print([
#     name for name in dir(weather_client)
#     if "upsert" in name.lower()
# ])

In [0]:

docs = weather_client.fetch_location_input(
    location="Denver, CO"
)

print(f"Documents fetched: {len(docs)}")

for doc in docs:
    print("\n---")
    print("Type:", doc["source_type"])
    print("Headline:", doc["headline"])
    print("Issued:", doc["issued_at"])
    print("Text length:", len(doc["narrative_text"] or ""))
    print("Preview:", (doc["narrative_text"] or "")[:300])

### keep the NWS fetching logic separate from the database-writing logic

In [0]:

# /weather/sync
#       ↓
# WeatherClient.fetch_location_input()
#       ↓
# normalized documents
#       ↓
# upsert_documents()
#       ↓
# weather_documents

In [0]:

docs = weather_client.fetch_location_input(
    location="Denver, CO"
)
print(f"Fetched: {len(docs)} documents")
synced = weather_module.upsert_documents(docs)
print(f"Synced: {synced} documents")

### Endpoint implementation

In [0]:
# %pip install flask

In [0]:
app_module = load_module("app")

print("app loaded successfully")

In [0]:
client = app_module.app.test_client()

response = client.post(
    "/weather/sync",
    json={
        "locations": ["Chicago, IL"],
        "limit": 50,
    },
)

print("Status:", response.status_code)
print("Response:", response.get_json())

In [0]:
response = client.post(
    "/weather/sync",
    json={
        "locations": [
            "Chicago, IL",
            "Austin, TX",
            "41.8781,-87.6298",
        ],
    },
)

print("Status:", response.status_code)
print("Response:", response.get_json())

In [0]:
rows = lakebase_client.run_query("""
    SELECT count(*)
    FROM weather_documents
""")

rows

In [0]:
# In practice, you have one database write mechanism:
# upsert_documents() → INSERT ... ON CONFLICT via psycopg2

# And two ways to trigger it:
# Directly from Python: weather_module.upsert_documents(docs). 
# But before that, you need to run: docs = weather_client.fetch_location_input(location="Denver, CO"). 
# The direct function call is useful for development/testing and for internal jobs.

# Through the REST API: POST /weather/sync. For the final project, the REST endpoint is the intended external interface. 

# An endpoint is usually described by: HTTP method + URL path
# So POST /weather/sync is an endpoint.
# And def sync_weather() is the Python function that handles that endpoint.

# An endpoint is a specific interface exposed by an application that allows a user, tool, or another application to interact with it over HTTP method. The request is a POST request, and the server sends an HTTP response back.